# Santolina tutor: LoRA fine-tune of Qwen3.5-0.8B

Before running:
1. On the laptop: `uv run python laptop/build_tutor_sft.py`, then upload `data/processed/tutor_sft/`
   as a **Private** Kaggle dataset (M3Kang is research-only: never make it, or the model, public).
2. Notebook settings: **Accelerator GPU T4 x2**, **Internet on**, add that dataset as input.
3. Run the cells up to the **dry run** interactively (about 5 min). It checks memory and prints an
   upper bound for the full training time.
4. Then **Save Version → Save & Run All (Commit)**: the run continues with the browser closed (max 12 h)
   and `/kaggle/working/` is kept as the version's output
   (`adapter/`, `merged/`, `qwen35-tutor-Q8_0.gguf`).

Knobs (env vars, set in the CONFIG cell): `EPOCHS` (2), `LR` (2e-4), `LORA_R` (32),
`BATCH`/`GRAD_ACCUM` (4/4), `MAX_MINUTES` (time limit: stop, then merge and save as usual).
If the dry run runs out of memory: `BATCH=2 GRAD_ACCUM=8`. If the loss turns NaN on T4: `FP16=0`.

In [ ]:
# flash-linear-attention: fast Triton kernels for Qwen3.5's Gated DeltaNet layers (else a slow
# PyTorch fallback). If training crashes inside fla/triton: !pip uninstall -y flash-linear-attention fla-core
!pip install -q -U "transformers>=5" peft accelerate safetensors flash-linear-attention
# Kaggle ships torchao 0.10; recent peft refuses to start when an old torchao is installed. Unused here.
!pip uninstall -y -q torchao
!nvidia-smi -L

In [ ]:
# CONFIG
import os
os.environ.update({"EPOCHS": "2", "LR": "2e-4", "LORA_R": "32", "BATCH": "4", "GRAD_ACCUM": "4"})

In [ ]:
import glob, os, subprocess
SCRIPT = glob.glob("/kaggle/input/**/train_tutor.py", recursive=True)[0]
os.environ["DATA_DIR"] = os.path.dirname(SCRIPT)
N_GPU = int(subprocess.check_output("nvidia-smi -L | wc -l", shell=True))
print(SCRIPT, N_GPU, "GPU(s)")

## Dry run: 3 steps on the longest examples (memory check + ETA)

In [ ]:
!DRY_RUN=1 torchrun --nproc_per_node {N_GPU} {SCRIPT}

## Full training

In [ ]:
!torchrun --nproc_per_node {N_GPU} {SCRIPT}

## Sanity check: a few replies on validation problems (outputs stay in Kaggle)

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
path = "/kaggle/working/merged"
tok = AutoTokenizer.from_pretrained(path)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32  # fp16 can overflow
model = AutoModelForCausalLM.from_pretrained(path, dtype=dtype).cuda().eval()
val = [json.loads(l) for l in open(os.path.join(os.environ["DATA_DIR"], "val.jsonl"))]
for task in ("prepare", "hint", "error_question", "free", "manipulation"):
    ex = next((r for r in val if r["task"] == task), None)
    if ex is None:  # never stop the notebook here: the GGUF cell comes next
        continue
    prompt = tok.apply_chat_template(ex["messages"][:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=500 if task == "prepare" else 120, do_sample=False)
    print(f"===== {task}: {ex['messages'][-2]['content'][:120]}")
    print(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
del model; torch.cuda.empty_cache()

## Convert to GGUF Q8_0 for the board (llama.cpp at the commit pinned in config.env)

In [ ]:
!git clone -q https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!cd /kaggle/working/llama.cpp && git checkout -q 2cbe0ec429b849a88664e0f62388fc92a5abd699
!pip install -q /kaggle/working/llama.cpp/gguf-py sentencepiece
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged --outtype q8_0 --outfile /kaggle/working/qwen35-tutor-Q8_0.gguf
# If the converter stops with an error about an MTP head, add --no-mtp to the line above.
!rm -rf /kaggle/working/llama.cpp /kaggle/working/ckpt
!ls -lh /kaggle/working /kaggle/working/ckpt 2>/dev/null